<a href="https://colab.research.google.com/github/simionRiccard0/Deep-Learning-Project/blob/main/viral_genomes_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DNA sequences for identifying viral
genomes in human samples (la falda)

In [1]:
!git clone https://github.com/NeuroCSUT/ViraMiner.git

Cloning into 'ViraMiner'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 78 (delta 10), reused 10 (delta 10), pack-reused 63 (from 1)
Receiving objects: 100% (78/78), 247.04 MiB | 15.76 MiB/s, done.
Resolving deltas: 100% (28/28), done.
Updating files: 100% (57/57), done.


In [2]:
import os
os.listdir('ViraMiner/data/DNA_data/')

['fullset_train.csv',
 'exp18_2015_F.csv',
 'exp12_2014_J1.csv',
 'exp3_2013_H4.csv',
 'exp10_2014_G6.csv',
 'exp2_2013_E2_SCC.csv',
 'create_LOO_set.py',
 'exp5_2014_D3.csv',
 'fullset_validation.csv',
 'exp15_2014_P.csv',
 'exp14_2014_O.csv',
 'exp1_2011_N19.csv',
 'exp11_2014_G7.csv',
 'create_dataset.py',
 'fullset_test.csv',
 'exp6_2014_E1.csv',
 'exp9_2014_G5.csv',
 'exp4_2014_B.csv',
 'exp17_2015_F2.csv',
 'exp0_2011_G5.csv',
 'exp7_2014_F1.csv',
 'exp8_2014_G1.csv',
 'exp13_2014_N1.csv',
 'exp16_2014_R1.csv']

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [4]:
import pandas as pd

train = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_train.csv',
    header=None
)

val = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_validation.csv',
    header=None
)

test = pd.read_csv(
    'ViraMiner/data/DNA_data/fullset_test.csv',
    header=None
)

print("Train shape:", train.shape)
print(train.head())

Train shape: (211239, 3)
                                                 0  \
0  seq1006848_2014_P_Lagheden_Matti-HPV-vacc-CIN31   
1                     seq4116690_2014_G6_Hultin_MS   
2            seq360982_2014_F1_PARAFFINBLANKBLOCKS   
3             seq1135576_2011_N19_VIRASKINFAPMISEQ   
4                        seq277_2014_G6_Hultin_MS8   

                                                   1  2  
0  CAAGCCAAGATTTTCTCGCGTCACACTACTCATGACCATTGTATTA...  0  
1  AACGAAGCACGGGCCGAGAGATTGAGGAACCAAGGTCCAGCTCTAG...  0  
2  TAGTGGGTGAGGTTTCTATTTCCATAATGATCTCGCCTCAATTACT...  0  
3  ATATGACCATTCTTGCAAGGTAACACAGGTACATTTTCACAAAGTG...  0  
4  GGTCTTAAAACAACAGAAATTTTTTCCATCACAGTTGCAGAAATTA...  0  


In [5]:
print("Sequence length:", len(train[1][0]))

print("\nClass balance:")
print(train[2].value_counts())

Sequence length: 300

Class balance:
2
0    206773
1      4466
Name: count, dtype: int64


DNA sequences encoding:

In [6]:
import torch
from torch.utils.data import Dataset
#DNA one-hot encoding
mapping = {
    "A": [1,0,0,0],
    "C": [0,1,0,0],
    "G": [0,0,1,0],
    "T": [0,0,0,1],
    "N": [0,0,0,0]
}

MAX_LEN = 300

def encode_sequence(seq):

    seq = seq[:MAX_LEN]

    encoded = [
        mapping.get(base, [0,0,0,0])
        for base in seq
    ]

    while len(encoded) < MAX_LEN:
        encoded.append([0,0,0,0])

    return encoded


class DNADataset(Dataset):

    def __init__(self, dataframe):

        self.sequences = dataframe[1].values
        self.labels = dataframe[2].values

    def __len__(self):

        return len(self.sequences)

    def __getitem__(self, idx):

        seq = self.sequences[idx]
        label = self.labels[idx]

        x = torch.tensor(
            encode_sequence(seq),
            dtype=torch.float32
        )

        y = torch.tensor(
            label,
            dtype=torch.float32
        )

        return x, y

Preparing the dataset (DataLoader):

In [7]:
from torch.utils.data import DataLoader

train_dataset = DNADataset(train)
val_dataset   = DNADataset(val)
test_dataset  = DNADataset(test)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 300, 4])
torch.Size([128])


CNN Branch

In [8]:
import torch.nn as nn
class CNNBranch(nn.Module):

    def __init__(self, pool_type="max"):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv1d(4, 64, kernel_size=10, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU()

        )

        if pool_type == "max":
            self.pool = nn.AdaptiveMaxPool1d(1)
        else:
            self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.conv(x)

        x = self.pool(x)

        return x.squeeze(-1)

Transformer branch

In [9]:
class TransformerBranch(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Linear(4, 128)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):

        x = self.embedding(x)

        x = self.transformer(x)

        x = x.permute(0, 2, 1)

        x = self.pool(x)

        return x.squeeze(-1)

ViraExplorer Model (CNN Pattern branch + CNN Frequency branch + Transformer)

In [10]:
class ViraExplorer(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn_max = CNNBranch("max")
        self.cnn_avg = CNNBranch("avg")

        self.transformer = TransformerBranch()

        self.fc = nn.Sequential(

            nn.Linear(256*2 + 128, 256),

            nn.BatchNorm1d(256),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(256, 1)

        )

    def forward(self, x):

        x1 = self.cnn_max(x)
        x2 = self.cnn_avg(x)
        x3 = self.transformer(x)

        x = torch.cat([x1, x2, x3], dim=1)

        return self.fc(x)

Focal Loss

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):

    def __init__(self, alpha=0.25, gamma=2):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):

        targets = targets.unsqueeze(1)

        BCE_loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

        pt = torch.exp(-BCE_loss)

        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss

        return focal_loss.mean()

Training Loop

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = ViraExplorer().to(device)

criterion = FocalLoss(
    alpha=0.25,
    gamma=2
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

scaler = torch.cuda.amp.GradScaler()

from sklearn.metrics import roc_auc_score

best_auc = 0
patience = 5
counter = 0

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    patience=3,
    factor=0.5
)

EPOCHS = 40

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}, "
        f"Loss: {avg_loss:.4f}"
    )

    # Validation
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            outputs = model(X_batch)

            probs = torch.sigmoid(outputs)

            preds.extend(
                probs.cpu().numpy().flatten()
            )

            targets.extend(
                y_batch.cpu().numpy()
            )

    auc = roc_auc_score(
        targets,
        preds
    )

    print(
        f"Validation AUC: {auc:.4f}"
    )

    scheduler.step(auc)

    if auc > best_auc:

        best_auc = auc
        counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

    else:

        counter += 1

        if counter >= patience:

            print("Early stopping triggered")
            break

CUDA available: True
GPU: Tesla T4


/tmp/ipykernel_752/1002712274.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1, Loss: 0.0092
Validation AUC: 0.8509
Epoch 2, Loss: 0.0057
Validation AUC: 0.8657
Epoch 3, Loss: 0.0049
Validation AUC: 0.8752
Epoch 4, Loss: 0.0041
Validation AUC: 0.8685
Epoch 5, Loss: 0.0033
Validation AUC: 0.8634
